In [ ]:
# Langkah 1: Upload dan baca data Excel
from google.colab import files
import pandas as pd
import numpy as np
import scipy.stats as stats

# Upload file
uploaded = files.upload()

# Membaca file Excel yang diunggah
filename = list(uploaded.keys())[0]
df = pd.read_excel(filename)

# Pastikan hanya ambil kolom yang dibutuhkan
X = df[['X1', 'X2', 'X3', 'X4', 'X5']]
y = df['Y']

# Langkah 2: Fungsi regresi OLS
def fit_ols(X, y):
    X = np.column_stack([np.ones(X.shape[0]), X])  # Tambahkan intercept
    XtX = np.dot(X.T, X)
    XtX_inv = np.linalg.inv(XtX)
    Xty = np.dot(X.T, y)
    beta = np.dot(XtX_inv, Xty)

    y_pred = np.dot(X, beta)
    residuals = y - y_pred
    mse = np.sum(residuals**2) / (X.shape[0] - X.shape[1])
    se = np.sqrt(np.diag(XtX_inv) * mse)
    t_stats = beta / se
    p_values = [2 * (1 - stats.t.cdf(np.abs(t), X.shape[0] - X.shape[1])) for t in t_stats]

    return beta, se, t_stats, p_values, residuals

# Langkah 3: Regresi Bertahap
def stepwise_regression(X, y, p_enter=0.05, p_exit=0.1):
    included = []
    while True:
        changed = False

        # Langkah maju
        excluded = list(set(X.columns) - set(included))
        best_pval = p_enter
        best_feature = None

        for feature in excluded:
            candidates = included + [feature]
            X_subset = X[candidates].values
            _, _, _, p_values, _ = fit_ols(X_subset, y)
            pval = p_values[-1]  # P-value dari variabel baru

            if pval < best_pval:
                best_pval = pval
                best_feature = feature

        if best_feature is not None:
            included.append(best_feature)
            changed = True

        # Langkah mundur
        if len(included) > 0:
            X_subset = X[included].values
            _, _, _, p_values, _ = fit_ols(X_subset, y)
            p_values = p_values[1:]  # Skip intercept

            max_pval = np.max(p_values)
            if max_pval > p_exit:
                idx = np.argmax(p_values)
                removed = included.pop(idx)
                changed = True

        if not changed:
            break

    return included

# Langkah 4: Eksekusi regresi bertahap
selected_vars = stepwise_regression(X, y)
print(f"Variabel terpilih: {selected_vars}")

# Langkah 5: Model akhir
X_final = X[selected_vars].values
beta, se, t_stats, p_values, residuals = fit_ols(X_final, y)

coef_table = pd.DataFrame({
    'Variabel': ['Intercept'] + selected_vars,
    'Koefisien': beta,
    'Std Error': se,
    't-stat': t_stats,
    'p-value': p_values
})

print("\nTabel Koefisien Regresi:")
print(coef_table)

# Langkah 6: Analisis Model
X_with_intercept = np.column_stack([np.ones(X_final.shape[0]), X_final])
y_pred = np.dot(X_with_intercept, beta)
SSE = np.sum((y - y_pred)**2)
SST = np.sum((y - np.mean(y))**2)
r_squared = 1 - SSE / SST
adj_r_squared = 1 - (SSE / (len(y) - len(beta))) / (SST / (len(y) - 1))
mse = SSE / (len(y) - len(beta))

print(f"\nR-squared: {r_squared:.4f}")
print(f"Adjusted R-squared: {adj_r_squared:.4f}")
print(f"MSE: {mse:.4f}")

# Uji F
F_stat = (r_squared / (len(beta) - 1)) / ((1 - r_squared) / (len(y) - len(beta)))
p_val_F = 1 - stats.f.cdf(F_stat, len(beta) - 1, len(y) - len(beta))
print(f"\nF-statistic: {F_stat:.4f}, p-value: {p_val_F:.6f}")

# Analisis residual
print("\nAnalisis Residual:")
print(f"Mean residual: {np.mean(residuals):.6f}")
print(f"Standard deviasi residual: {np.std(residuals):.4f}")

# Uji normalitas residual
shapiro_test = stats.shapiro(residuals)
print(f"\nUji Normalitas Residual (Shapiro-Wilk):")
print(f"Statistic: {shapiro_test.statistic:.4f}, p-value: {shapiro_test.pvalue:.4f}")

# Langkah 7: Persamaan regresi
equation = "Y = "
for i, var in enumerate(coef_table['Variabel']):
    sign = '+' if coef_table['Koefisien'][i] >= 0 and i > 0 else ''
    equation += f"{sign}{coef_table['Koefisien'][i]:.4f}"
    if i > 0:
        equation += f"*{var}"
equation += " + ε"
print("\nPersamaan Regresi Final:")
print(equation)


Saving data (2).xlsx to data (2).xlsx
Variabel terpilih: ['X3', 'X1', 'X5', 'X2', 'X4']

Tabel Koefisien Regresi:
    Variabel  Koefisien  Std Error     t-stat       p-value
0  Intercept  -0.604554   2.179055  -0.277439  7.827428e-01
1         X3   5.053872   0.210716  23.984246  0.000000e+00
2         X1   2.833722   0.176314  16.072051  0.000000e+00
3         X5   2.705231   0.255350  10.594225  1.092459e-13
4         X2   2.239443   0.332374   6.737712  2.773599e-08
5         X4   1.556967   0.553407   2.813422  7.300152e-03

R-squared: 0.9530
Adjusted R-squared: 0.9476
MSE: 1.1254

F-statistic: 178.3117, p-value: 0.000000

Analisis Residual:
Mean residual: -0.000000
Standard deviasi residual: 0.9952

Uji Normalitas Residual (Shapiro-Wilk):
Statistic: 0.9683, p-value: 0.1980

Persamaan Regresi Final:
Y = -0.6046+5.0539*X3+2.8337*X1+2.7052*X5+2.2394*X2+1.5570*X4 + ε
